# Importing modules
Modules need to be imported individually, we can then switch between cases by providing the correct modules to the Hamilton driver

## Execution and calculation of the qfi

In [6]:
from hamilton import driver
from hamilton.execution import executors

import rayfish
import rayfish.quantum_fisher_information as qfi
from rayfish.parameter_catalogue import common_transforms, distance_2_sources
from rayfish.quantum_fisher_information import incoherent_sources

dr = (
    driver.Builder()
    .enable_dynamic_execution(allow_experimental_mode=True)
    .with_modules(
        rayfish, qfi, incoherent_sources, common_transforms, distance_2_sources
    )
    .with_config({"dimension": 1})
    .with_local_executor(executors.SynchronousLocalTaskExecutor())
    .build()
)

# dr.visualize_execution(inputs={
#         "wave_vector": 2.0,
#         "rayleigh_length": 1.0,
#         "number_of_point_source_distributions": 5,
#         "maximal_length": 1.0,
#         "minimal_spacing": 0.1,
#     },
#     final_vars=["lyapunov_covariance_matrix_for_entire_distribution", "spacing"],
#     output_file_path="dag.png"
#     )

result = dr.execute(
    inputs={
        "wave_vector": 2.0,
        "rayleigh_length": 1.0,
        "number_of_point_source_distributions": 20,
        "maximal_length": 4.0,
        "minimal_spacing": 0.1,
    },
    # final_vars=["lyapunov_covariance_matrix_for_entire_distribution", "spacing"],
    final_vars=["coordinates", "spacing"],
)

## Plotting the results

In [3]:
import numpy as np
import pandas as pd
import plotly.express as px

qfi = np.real(result["lyapunov_covariance_matrix_for_entire_distribution"])
gamma = np.imag(result["lyapunov_covariance_matrix_for_entire_distribution"])
x = result["spacing"]
dfs = {
    "spacing": x,
    "var_xy": qfi[:, 0, 0],
    "var_z": qfi[:, 1, 1],
    "cov_xyz_1": qfi[:, 0, 1],
    "cov_xyz_2": qfi[:, 1, 0],
    "gamma_xy": gamma[:, 0, 0],
    "gamma_z": gamma[:, 1, 1],
    "cov_gamma_1": gamma[:, 0, 1],
    "cov_gamma_2": gamma[:, 1, 0],
}

df = pd.DataFrame.from_dict(dfs)
df = df.set_index("spacing")
fig = px.scatter(df)
fig.show()

ModuleNotFoundError: No module named 'plotly'